# Exploratory EDA

We have a dataset consisting of a total 16,820 records across 22 columns, which include earthquake features such as time, space, depth, magnitude, among others.

A brief description of the data shows us the dataset isn't Japan-specific, and includes other geographic regions such as China, Russia, and the Kuril Islands.

Missing values comprise the following columns and total null values on each: nst with 5069 missing records, gap with 1323 missing records, dmin with 9279, rms with 81, horizontalError 10002, depthError with 4354, magError with 9444, and magNst with 2773. The remaining features-time, latitude, longitude, depth, mag, magType, net, id, updated, place, type, status, locationSource, magSource-have no null values. However, because of these might be identification features and metadata, they might be removed from the dataset, along with null values.

There were no duplicates in this dataset.

The earliest recorded data was from 2000-01-03 18:01:26.370000+00:00, and the latest 2026-09-05 00:22:45.357000+00:00; minimum latitude is 24.4026 and maximum is 45.594; Maximum and Minimum longitude 122.917 and 153.782 respectively; minimum magnitude in the dataset is M 4.5 and maximum recorded is M 9.1; minimum depth 0 and maximum depth 683.36 (need to verify if miles or kms).

Categorical values show us 8 different categories of magnitude and identified the two types of events: earthquakes with a total of 16,815 records, and nuclear explosions with a total of 5 records.

In [ ]:
#import pandas as pd
import numpy as np

df = pd.read_csv('/content/Japan earthquakes 2000 - 2026.csv')

In [ ]:
#print("---DATA SHAPE---")
print(df.shape)

print("\n---COLUMNS---")
print(df.columns)

print("\n---HEAD---")
print(df.head())

print("---INFO---")
print(df.info())

print("\n---DESCRIBE---")
print(df.describe)

print("\n---MISSING VALUES---")
print(df.isnull().sum())

print("\n---DUPLICATE COUNT---")
print(df.duplicated().sum())

print("\n---TIME RANGE---")
df['time'] = pd.to_datetime(df['time'])

print("Time range:")
print("Earliest:", df['time'].min())
print("Latest:", df['time'].max())

print("\n---LATITUDE/LONGITUDE RANGE---")
print("Latitude range:")
print("Minimum:", df['latitude'].min())
print("Maximum:", df['latitude'].max())

print("\nLongitude range:")
print("Minimum:", df['longitude'].min())
print("Maximum:", df['longitude'].max())

print("\n---MAGNITUDE RANGE---")
print("Magnitude range:")
print("Minimum:", df['mag'].min())
print("Maximum:", df['mag'].max())

print("\n---DEPTH RANGE---")
print("Depth range:")
print("Minimum:", df['depth'].min())
print("Maximum:", df['depth'].max())

print("\n---UNIQUE VALUES---")
print(df.nunique())

print("\n---CATEGORICAL VALUES---")
for col in df.select_dtypes(include='object').columns:
  print(f"\n{col}:")
  print(df[col].value_counts().head(10))

---DATA SHAPE---
(16820, 22)

---COLUMNS---
Index(['time', 'latitude', 'longitude', 'depth', 'mag', 'magType', 'nst',
       'gap', 'dmin', 'rms', 'net', 'id', 'updated', 'place', 'type',
       'horizontalError', 'depthError', 'magError', 'magNst', 'status',
       'locationSource', 'magSource'],
      dtype='object')

---HEAD---
                       time  latitude  longitude    depth  mag magType    nst  \
0  2026-09-05T00:22:45.357Z   40.6426   143.5636   33.857  4.5      mb   18.0   
1  2026-09-04T23:52:29.825Z   44.9418   149.9838   32.000  5.5     mww  105.0   
2  2026-09-01T19:21:23.337Z   41.6191   142.0502   74.021  4.6      mb   85.0   
3  2026-08-31T09:19:57.841Z   45.1453   148.2193  147.811  4.6      mb   62.0   
4  2026-08-30T00:14:19.915Z   42.0423   142.6167   63.625  4.5      mb   41.0   

     gap   dmin   rms  ...                   updated  \
0  177.0  1.405  0.53  ...  2026-09-05T00:50:28.040Z   
1   82.0  5.348  1.19  ...  2026-09-05T23:59:01.035Z   
2  114.0  0.

# PHASE 2: DATA CLEANING
Phase 2 is where the raw catalog is transformed into a clean earthquake dataset representing the geographic region the study is aimed at.

The first step is defining the geographic boundary by establishing the latitude and longitude bounds that will be used for the Japan dataset. We will also be removing non-earthquake events. From the preliminary data exploration, it was found from the categorical values that five of the records from the dataset are from nuclear explosions. Therefore, a filter will be added to include only events categorized as 'earthquakes'.

In [ ]:
## Remove non-earthquake events
df = df[df['type'] == 'earthquake'].copy()

print(df.shape)
print(df['type'].value_counts())

(16815, 22)
type
earthquake    16815
Name: count, dtype: int64


Next, we want to handle the geographic contamination by temporarily focusing on the *place* feature to show exactly what the contamination is specifically.



In [ ]:
print(df['place'].value_counts().head(30))

place
Izu Islands, Japan region                1454
Bonin Islands, Japan region              1368
Volcano Islands, Japan region             262
off the east coast of Honshu, Japan       188
near the east coast of Honshu, Japan       87
east of the Kuril Islands                  49
Kuril Islands                              41
Ryukyu Islands, Japan                      37
Hokkaido, Japan region                     23
eastern Honshu, Japan                      15
near the south coast of Honshu, Japan      14
238 km ESE of Ishinomaki, Japan            14
61 km SSE of Shimoda, Japan                12
66 km SSE of Shimoda, Japan                11
71 km SSE of Shimoda, Japan                11
119 km E of Miyako, Japan                  10
Kyushu, Japan                              10
67 km SSE of Shimoda, Japan                10
233 km ESE of Ishinomaki, Japan            10
236 km ESE of Ishinomaki, Japan            10
120 km E of Miyako, Japan                  10
Sea of Japan                

In [ ]:
print(df.loc[~df['place'].str.contains('Japan', case=False, na=False),
             ['latitude', 'longitude', 'place']].head(30))

     latitude  longitude                           place
1     44.9418   149.9838  168 km ESE of Kuril’sk, Russia
3     45.1453   148.2193   28 km ESE of Kuril’sk, Russia
7     44.8019   149.9582  170 km ESE of Kuril’sk, Russia
13    44.6862   146.4404      82 km NE of Otrada, Russia
32    43.8872   147.5980     71 km E of Shikotan, Russia
44    43.1851   146.9239   70 km SSE of Shikotan, Russia
45    43.1737   147.0047   73 km SSE of Shikotan, Russia
67    44.4561   149.1226   130 km SE of Kuril’sk, Russia
77    44.5094   147.8897     79 km S of Kuril’sk, Russia
81    43.9984   146.5781   24 km NNW of Shikotan, Russia
100   45.4859   151.2373    264 km E of Kuril’sk, Russia
101   45.4715   151.4830    283 km E of Kuril’sk, Russia
113   44.2553   148.4830  118 km SSE of Kuril’sk, Russia
127   43.5820   135.8435      48 km ESE of Ol’ga, Russia
129   43.5389   147.3424   57 km ESE of Shikotan, Russia
130   44.5924   149.4805  144 km ESE of Kuril’sk, Russia
146   44.0161   147.8463   93 k

The process above show us the records outside of the Japan region by filtering through the *place* feature. However, the data not containing "Japan" should not be deleted yet, as *place* is a text description generated by the catalog and not the best geographic criterion.

By quantifying the contamination next, we can determine the best approach and features for our geographic criterion:

In [ ]:
# quantify contamination to determine geographic data
non_japan = df[
    ~df['place'].str.contains('Japan', case=False, na=False)
]

print("Non-Japan-labeled events:", len(non_japan))
print("Percentage of dataset:", len(non_japan) / len(df) * 100)

print("\nCoordinate ranges of non-Japan-labeled events:")
print("Latitude:", non_japan['latitude'].min(), "to", non_japan['latitude'].max())
print("Longitude:", non_japan['longitude'].min(), "to", non_japan['longitude'].max())

print(non_japan['place'].value_counts().head(20))

Non-Japan-labeled events: 1344
Percentage of dataset: 7.992863514719001

Coordinate ranges of non-Japan-labeled events:
Latitude: 24.938 to 45.594
Longitude: 122.9347 to 153.782
place
east of the Kuril Islands         49
Kuril Islands                     41
81 km SSE of Kuril’sk, Russia      7
226 km E of Kuril’sk, Russia       7
125 km SE of Kuril’sk, Russia      6
146 km SE of Kuril’sk, Russia      5
144 km SE of Kuril’sk, Russia      5
129 km SSE of Kuril’sk, Russia     4
139 km SE of Kuril’sk, Russia      4
76 km SSE of Kuril’sk, Russia      4
167 km SE of Kuril’sk, Russia      4
133 km SE of Kuril’sk, Russia      4
123 km SE of Kuril’sk, Russia      4
128 km SSE of Kuril’sk, Russia     4
125 km ESE of Kuril’sk, Russia     4
122 km SE of Kuril’sk, Russia      4
159 km SE of Kuril’sk, Russia      4
162 km E of Kuril’sk, Russia       4
134 km SE of Kuril’sk, Russia      4
243 km E of Kuril’sk, Russia       4
Name: count, dtype: int64


In [ ]:
# Newly cleaned dataset
print("---CLEANED DATASET---")
print("Shape:", df.shape)

print("\n---TIME RANGE---")
print("Earliest:", df['time'].min())
print("Latest:", df['time'].max())

print("\n---COORDINATES---")
print("Latitude:", df['latitude'].min(), "to", df['latitude'].max())
print("Longitude:", df['longitude'].min(), "to", df['longitude'].max())

print("\n---MAGNITUDE---")
print("Magnitude:", df['mag'].min(), "to", df['mag'].max())

print("\n---DEPTH---")
print("Depth:", df['depth'].min(), "to", df['depth'].max())

print("\n---MISSING VALUES---")
print(df.isnull().sum())

print("\n---DUPLICATES---")
print(df.duplicated().sum())

---CLEANED DATASET---
Shape: (16815, 22)

---TIME RANGE---
Earliest: 2000-01-03 18:01:26.370000+00:00
Latest: 2026-09-05 00:22:45.357000+00:00

---COORDINATES---
Latitude: 24.4026 to 45.594
Longitude: 122.917 to 153.782

---MAGNITUDE---
Magnitude: 4.5 to 9.1

---DEPTH---
Depth: 0.0 to 683.36

---MISSING VALUES---
time                   0
latitude               0
longitude              0
depth                  0
mag                    0
magType                0
nst                 5066
gap                 1322
dmin                9276
rms                   81
net                    0
id                     0
updated                0
place                  0
type                   0
horizontalError    10000
depthError          4352
magError            9442
magNst              2773
status                 0
locationSource         0
magSource              0
dtype: int64

---DUPLICATES---
0


The geographic criterion is established based on coordinates. The data was screened using latitude and longitude instead of textual labels, because the study focuses on the spatial distribution of seismic events and the catalog includes offshore and geographically adjacent regions.

Base on the geographical analysis above we can determine that of 16,815 events (after removing the 5 nuclear explosion events):

* 1,344 events (7.99%) whose place field does not contain "Japan"
* 15,471 events (92.01%) whose place does contain "Japan".

The non-Japanese events span the same coordinates as the dataset:

* Latitude: 24.938-45.594
* Longitude: 122.9347-153.782

The next step then is to define a study area bounding box using coordinates corresponding to the Japan region. Since we are using USGS data, the best approach is using a geographic reference for Japan's extent.

In [ ]:
# Geographic Bounding Box Diagnostic
print("Events outside proposed Japan latitude range:")
print(df[(df['latitude'] < 24) | (df['latitude'] > 46)].shape)

print("\nEvents outside proposed Japan longitude range:")
print(df[(df['longitude'] < 122) | (df['longitude'] > 154)].shape)

Events outside proposed Japan latitude range:
(0, 22)

Events outside proposed Japan longitude range:
(0, 22)


In [ ]:
outside = df[
    (df['latitude'] < 24) |
    (df['latitude'] > 46) |
    (df['longitude'] < 122) |
    (df['longitude'] > 154)
]

print("Total outside bounding box:", len(outside))
print(outside[['latitude', 'longitude', 'place']].head(30))

Total outside bounding box: 0
Empty DataFrame
Columns: [latitude, longitude, place]
Index: []


The above exercise confirms that none of the 16,815 earthquake records fall outside the proposed bounding box. Therefore, nothing is removed based on the bounding box.

Because the non-Japan records include the Kuril region and offshore areas spatially adjacent to Japan, they remain inside the geographic study region.

That said, for the model, latitude and longitude remain the authoritative spatial variables.

In [ ]:
# Save cleaned dataset for ML modeling
df.to_csv('/content/Japan_earthquakes_cleaned.csv', index=False)

print("Cleaned dataset saved.")
print("Shape:", df.shape)

Cleaned dataset saved.
Shape: (16815, 22)
